# Lab 06 External V2 — Secure-View Governance

This notebook implements governance without attaching native row filters or
column masks to the externally path-written Gold base tables.

It creates:

- `vw_fact_encounters_secure` for organization-based row-level security;
- `vw_dim_patient_secure` for masking patient-sensitive columns;
- `lab06_user_organization_access` as the RLS mapping table;
- `lab06_patient_data_privileged_users` as the CLS privilege mapping table.

`run_demo=false` is the safe default for recurring Job execution.
Set `run_demo=true` when you want the temporary RLS/CLS demonstration and PASS output.


In [0]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "02 External V2 target schema",
)
ensure_dropdown_widget(
    "run_demo",
    "false",
    ["true", "false"],
    "03 Run restricted/masked demo",
)

catalog = dbutils.widgets.get("catalog").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
run_demo = dbutils.widgets.get("run_demo").strip().lower() == "true"

target_schema_fqn = f"{catalog}.{target_schema}"

print(f"Catalog       : {catalog}")
print(f"Target schema : {target_schema}")
print(f"Run demo      : {run_demo}")


In [0]:
from pyspark.sql import functions as F

fact_encounters = f"{target_schema_fqn}.fact_encounters"
dim_patient = f"{target_schema_fqn}.dim_patient"

secure_fact_view = f"{target_schema_fqn}.vw_fact_encounters_secure"
secure_patient_view = f"{target_schema_fqn}.vw_dim_patient_secure"

org_access_table = f"{target_schema_fqn}.lab06_user_organization_access"
patient_privileged_table = (
    f"{target_schema_fqn}.lab06_patient_data_privileged_users"
)

sensitive_columns = ["ssn", "first_name", "last_name", "address"]

for table_name in [fact_encounters, dim_patient]:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Required Gold table does not exist: {table_name}")

fact_columns = set(spark.table(fact_encounters).columns)
patient_columns = spark.table(dim_patient).columns

if "organization_id" not in fact_columns:
    raise RuntimeError(
        f"{fact_encounters} is missing required column organization_id"
    )

missing_sensitive = sorted(
    set(sensitive_columns) - set(patient_columns)
)
if missing_sensitive:
    raise RuntimeError(
        "dim_patient is missing sensitive columns: "
        + ", ".join(missing_sensitive)
    )

session_user = spark.sql(
    "SELECT SESSION_USER() AS user"
).first()["user"]

escaped_user = session_user.replace("'", "''")

print(f"Session user        : {session_user}")
print(f"Fact base table     : {fact_encounters}")
print(f"Patient base table  : {dim_patient}")
print(f"Secure fact view    : {secure_fact_view}")
print(f"Secure patient view : {secure_patient_view}")


## Remove legacy native policies if present

External V2 writes Delta files by explicit ABFSS paths. Native table row filters
or column masks make path-based access unsupported, so any legacy native policies
are removed before the secure-view layer is built.


In [0]:
legacy_cleanup_statements = [
    f"ALTER TABLE {fact_encounters} DROP ROW FILTER",
    *[
        (
            f"ALTER TABLE {dim_patient} "
            f"ALTER COLUMN `{column_name}` DROP MASK"
        )
        for column_name in sensitive_columns
    ],
]

for statement in legacy_cleanup_statements:
    try:
        spark.sql(statement)
        print(f"Applied legacy cleanup: {statement}")
    except Exception:
        print(f"Legacy cleanup not required: {statement}")

print("Base Gold tables remain compatible with path-based writes.")


## Capture baseline and select an organization for the demo


In [0]:
baseline_rows = spark.table(fact_encounters).count()

org_counts_df = (
    spark.table(fact_encounters)
    .filter(F.col("organization_id").isNotNull())
    .groupBy("organization_id")
    .count()
    .orderBy(F.desc("count"))
)

demo_org = org_counts_df.first()

if demo_org is None:
    raise RuntimeError(
        "fact_encounters contains no organization_id value for the RLS demo"
    )

demo_organization_id = str(demo_org["organization_id"])
expected_restricted_rows = int(demo_org["count"])

print(f"Baseline encounters      : {baseline_rows}")
print(f"Demo organization        : {demo_organization_id}")
print(f"Expected restricted rows : {expected_restricted_rows}")


## Create or repair governance mapping tables

Older Lab 06 versions may have helper tables with incompatible schemas.
If that is detected, only the stale helper table is recreated. Gold business
tables are never dropped here.


In [0]:
def ensure_mapping_table(
    table_name: str,
    create_sql: str,
    required_columns: set[str],
) -> None:
    if spark.catalog.tableExists(table_name):
        existing_columns = {
            field.name.lower()
            for field in spark.table(table_name).schema.fields
        }

        if not required_columns.issubset(existing_columns):
            print(f"Recreating stale governance table: {table_name}")
            print(f"Existing columns: {sorted(existing_columns)}")

            # A prior secure view may reference this helper table.
            if table_name == org_access_table:
                spark.sql(f"DROP VIEW IF EXISTS {secure_fact_view}")
            if table_name == patient_privileged_table:
                spark.sql(f"DROP VIEW IF EXISTS {secure_patient_view}")

            spark.sql(f"DROP TABLE {table_name}")

    spark.sql(create_sql)


ensure_mapping_table(
    org_access_table,
    f'''
    CREATE TABLE IF NOT EXISTS {org_access_table} (
        user_email STRING,
        organization_id STRING
    )
    USING DELTA
    ''',
    {"user_email", "organization_id"},
)

ensure_mapping_table(
    patient_privileged_table,
    f'''
    CREATE TABLE IF NOT EXISTS {patient_privileged_table} (
        user_email STRING
    )
    USING DELTA
    ''',
    {"user_email"},
)

spark.sql(
    f"DELETE FROM {org_access_table} "
    f"WHERE lower(user_email) = lower('{escaped_user}')"
)
spark.sql(
    f'''
    INSERT INTO {org_access_table}
    VALUES ('{escaped_user}', '*')
    '''
)

spark.sql(
    f"DELETE FROM {patient_privileged_table} "
    f"WHERE lower(user_email) = lower('{escaped_user}')"
)
spark.sql(
    f'''
    INSERT INTO {patient_privileged_table}
    VALUES ('{escaped_user}')
    '''
)

print("Governance mapping tables are ready.")
print("Current user initialized with full / privileged access.")


## Create the secure RLS view


In [0]:
spark.sql(
    f'''
    CREATE OR REPLACE VIEW {secure_fact_view} AS
    SELECT f.*
    FROM {fact_encounters} f
    WHERE EXISTS (
        SELECT 1
        FROM {org_access_table} a
        WHERE lower(a.user_email) = lower(SESSION_USER())
          AND (
              a.organization_id = '*'
              OR a.organization_id = CAST(f.organization_id AS STRING)
          )
    )
    '''
)

print(f"Created secure RLS view: {secure_fact_view}")


## Create the secure CLS view


In [0]:
projection = []

for column_name in patient_columns:
    quoted = f"`{column_name.replace('`', '``')}`"

    if column_name in sensitive_columns:
        projection.append(
            f'''
            CASE
                WHEN EXISTS (
                    SELECT 1
                    FROM {patient_privileged_table} ppu
                    WHERE lower(ppu.user_email) = lower(SESSION_USER())
                )
                THEN p.{quoted}
                ELSE CASE
                    WHEN p.{quoted} IS NULL THEN NULL
                    ELSE '***MASKED***'
                END
            END AS {quoted}
            '''.strip()
        )
    else:
        projection.append(f"p.{quoted}")

projection_sql = ",\n        ".join(projection)

spark.sql(
    f'''
    CREATE OR REPLACE VIEW {secure_patient_view} AS
    SELECT
        {projection_sql}
    FROM {dim_patient} p
    '''
)

print(f"Created secure CLS view: {secure_patient_view}")


## Validate RLS

With `run_demo=true`, the notebook temporarily maps the current user to one
organization, verifies the restricted count, and restores wildcard access.


In [0]:
rls_restricted_status = "SKIPPED"
rls_restored_status = "PASS"

if run_demo:
    spark.sql(
        f"DELETE FROM {org_access_table} "
        f"WHERE lower(user_email) = lower('{escaped_user}')"
    )

    escaped_demo_org = demo_organization_id.replace("'", "''")
    spark.sql(
        f'''
        INSERT INTO {org_access_table}
        VALUES ('{escaped_user}', '{escaped_demo_org}')
        '''
    )

    actual_restricted_rows = spark.table(secure_fact_view).count()

    rls_restricted_status = (
        "PASS"
        if actual_restricted_rows == expected_restricted_rows
        else "FAIL"
    )

    display(
        spark.createDataFrame(
            [
                (
                    baseline_rows,
                    expected_restricted_rows,
                    actual_restricted_rows,
                    rls_restricted_status,
                )
            ],
            [
                "baseline_rows",
                "expected_restricted_rows",
                "actual_restricted_rows",
                "status",
            ],
        )
    )

    spark.sql(
        f"DELETE FROM {org_access_table} "
        f"WHERE lower(user_email) = lower('{escaped_user}')"
    )
    spark.sql(
        f'''
        INSERT INTO {org_access_table}
        VALUES ('{escaped_user}', '*')
        '''
    )

restored_rows = spark.table(secure_fact_view).count()
rls_restored_status = (
    "PASS" if restored_rows == baseline_rows else "FAIL"
)

print(f"RLS restricted demo         : {rls_restricted_status}")
print(f"RLS full-access restoration : {rls_restored_status}")


## Validate CLS

With `run_demo=true`, the notebook temporarily removes the current user from
the privileged mapping, displays masked patient values, validates the masks,
and then restores privileged access.


In [0]:
cls_masked_status = "SKIPPED"
cls_restored_status = "PASS"

if run_demo:
    spark.sql(
        f"DELETE FROM {patient_privileged_table} "
        f"WHERE lower(user_email) = lower('{escaped_user}')"
    )

    masked_sample = (
        spark.table(secure_patient_view)
        .select("patient_id", *sensitive_columns)
        .limit(5)
    )
    display(masked_sample)

    mask_failures = 0

    for column_name in sensitive_columns:
        mask_failures += (
            spark.table(secure_patient_view)
            .filter(
                F.col(column_name).isNotNull()
                & (F.col(column_name) != F.lit("***MASKED***"))
            )
            .limit(1)
            .count()
        )

    cls_masked_status = (
        "PASS" if mask_failures == 0 else "FAIL"
    )

    spark.sql(
        f'''
        INSERT INTO {patient_privileged_table}
        VALUES ('{escaped_user}')
        '''
    )

remaining_masked_values = 0

for column_name in sensitive_columns:
    remaining_masked_values += (
        spark.table(secure_patient_view)
        .filter(
            F.col(column_name) == F.lit("***MASKED***")
        )
        .limit(1)
        .count()
    )

cls_restored_status = (
    "PASS" if remaining_masked_values == 0 else "FAIL"
)

print(f"CLS masked demo             : {cls_masked_status}")
print(f"CLS privileged restoration  : {cls_restored_status}")


## Final governance validation

Normal recurring runs use `run_demo=false`: the secure views are created and
the current Job identity remains on full/privileged access.

For evidence runs, set `run_demo=true`; all four checks should show `PASS`.


In [0]:
final_checks = [
    ("RLS restricted demo", rls_restricted_status),
    ("RLS full-access restoration", rls_restored_status),
    ("CLS masked demo", cls_masked_status),
    ("CLS privileged restoration", cls_restored_status),
]

final_validation_df = spark.createDataFrame(
    final_checks,
    ["governance_check", "status"],
)

display(final_validation_df)

failed_checks = [
    check_name
    for check_name, status in final_checks
    if status == "FAIL"
]

if failed_checks:
    raise RuntimeError(
        "External V2 secure-view governance validation failed: "
        + ", ".join(failed_checks)
    )

print("LAB 06 EXTERNAL V2 — SECURE-VIEW GOVERNANCE COMPLETE")
print(f"RLS view  : {secure_fact_view}")
print(f"CLS view  : {secure_patient_view}")
print(f"Principal : {session_user}")
print("Current user restored to full / privileged access.")
print("Base Gold tables remain free of native row filters/masks.")


## Consumer access pattern

For a real consumer or group:

1. add its organization mapping(s) to `lab06_user_organization_access`;
2. add it to `lab06_patient_data_privileged_users` only if unmasked patient data is allowed;
3. grant `USE CATALOG`, `USE SCHEMA`, and `SELECT` on the secure views;
4. do **not** grant direct `SELECT` on the governed base tables.

The recurring Gold Job keeps base-table access and remains compatible with
path-based External V2 writes.
